In [1]:
from jupyter_client import session
%pip install anthropic==0.120.2 python-dotenv==1.2.2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

In [3]:
import anthropic
client = anthropic.Anthropic(api_key=claude_api_key)

In [6]:
agent = client.beta.agents.create(
    name="Demo-Managed-Agent",
    model=claude_model_name,
    system = "You are a helpful AI Assistant",
    tools=[
        {            "type": "agent_toolset_20260401"
        },
    ],
)
print(f"Agent ID: {agent.id}, version: {agent.version}")

Agent ID: agent_01NjdPAXML9v9PrzZfWP23Sg, version: 1


In [7]:
environment = client.beta.environments.create(
    name="Demo-Managed-Environment",
    config= {
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)
print(f"Environment ID: {environment.id}")

Environment ID: env_01TuuChjmq9DQEwwL8eYYFv2


In [9]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Demo Managed Session",
)
print(f"Session ID: {session.id}")

Session ID: sesn_01VGJsPob525rxXeoqM4ZhZq


In [10]:
chat = True

while chat:
    user_query = input("Enter ""exit"" or your user query to continue")

    if user_query == "exit":
        chat = False
    else:
        with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": user_query,
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break

Hi Srinivas, nice to meet you too!

I'm Claude, an AI assistant. I can help with all sorts of things — writing and debugging code, working with files, research, data processing, automation, documentation, and more.

What can I help you with today?

Agent finished.
Goodbye, Srinivas! Feel free to come back anytime you need help. 👋

Agent finished.
